In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
# Notebook の変数を完全リセット
%reset -f

In [3]:
#ライブラリの読み込み
import numpy as np
import pandas as pd
import os
import pickle
import gc 

# 分布確認
# import pandas_profiling as pdp
import ydata_profiling as pdp # ライブラリ名称が変更になったため

# 可視化
import matplotlib.pyplot as plt

# 前処理
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder

# バリデーション
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold

# 評価指標
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# モデリング: lightgbm
import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

# matplotilbで日本語表示したい場合はこれをinstallしてインポートする
!pip install japanize-matplotlib
import japanize_matplotlib
%matplotlib inline

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 47.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for japanize-matplotlib: filename=japanize_matplotlib-1.1.3-py3-none-any.whl size=4120257 sha256=99cb86fd19698ab01b80d92e273094015ccf0a58590f1a62feb1d0e8e6e023f8
  Stored in directory: /root/.cache/pip/wheels/c1/f7/9b/418f19a7b9340fc16e071e89efc379aca68d40238b258df53d
Successfully built japanize-matplotlib


In [4]:
#ファイルの読み込み
df_train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df_test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
df_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [6]:
#レコード数とカラム数の確認
print(df_train.shape)

print(df_test.shape)

(891, 12)
(418, 11)


In [7]:
#データの確認
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [8]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


In [9]:
#欠損値の確認
df_train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [10]:
df_test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [11]:
median_age = df_train["Age"].median()

df_train["Age"] = df_train["Age"].fillna(median_age)
df_test["Age"]  = df_test["Age"].fillna(median_age)

In [12]:
median_fare = df_train["Fare"].median()

df_train["Fare"] = df_train["Fare"].fillna(median_fare)
df_test["Fare"]  = df_test["Fare"].fillna(median_fare)

In [13]:
mode_embarked = df_train["Embarked"].mode()[0]

df_train["Embarked"] = df_train["Embarked"].fillna(mode_embarked)
df_test["Embarked"]  = df_test["Embarked"].fillna(mode_embarked)

In [14]:
df_train_raw = df_train.copy()
df_test_raw = df_test.copy()

In [15]:
df_train = df_train.drop(columns=["Cabin"])
df_test  = df_test.drop(columns=["Cabin"])

In [16]:
df_train = df_train.drop(columns=["Name", "Ticket"])
df_test  = df_test.drop(columns=["Name", "Ticket"])

In [17]:
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

In [18]:
x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

In [19]:
for col in ["Sex", "Embarked"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

In [20]:
print(x_train.shape, y_train.shape, id_train.shape)

(891, 7) (891,) (891,)


In [21]:
#モデル学習、評価の関数化

params = {
    'boosting_type': 'gbdt',
    'objective': 'binary', 
    'metric': 'auc',
    'learning_rate': 0.1,
    'num_leaves': 16,
    'n_estimators': 5000,
    "random_state": 123,
    "importance_type": "gain",
}    

def train_cv(input_x, input_y, params, n_splits=5, run_name="run0"):

    # --- run ごとの保存フォルダ ---
    save_dir = f"{BASE_DIR}/{run_name}"
    os.makedirs(save_dir, exist_ok=True)
    

    metrics = []
    imp = pd.DataFrame()
    oof = np.zeros(len(input_x))

    cv = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123).split(input_x, input_y))

    for nfold in range(n_splits):
        idx_tr, idx_va = cv[nfold]
        x_tr, y_tr = input_x.iloc[idx_tr, :], input_y.iloc[idx_tr]
        x_va, y_va = input_x.iloc[idx_va, :], input_y.iloc[idx_va]

        model = lgb.LGBMClassifier(**params)

        model.fit(
            x_tr, y_tr,
            eval_set=[(x_tr, y_tr), (x_va, y_va)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
            ],
        )

        # --- run ごとにモデル保存 ---
        model.booster_.save_model(f"{save_dir}/model_fold{nfold}.txt")

        # AUC 用の確率予測
        y_tr_pred = model.predict_proba(x_tr)[:, 1]
        y_va_pred = model.predict_proba(x_va)[:, 1]

        auc_tr = roc_auc_score(y_tr, y_tr_pred)
        auc_va = roc_auc_score(y_va, y_va_pred)

        metrics.append([nfold, auc_tr, auc_va])

        oof[idx_va] = y_va_pred

        _imp = pd.DataFrame({
            "col": input_x.columns,
            "imp": model.feature_importances_,
            "nfold": nfold
        })
        imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

    metrics = np.array(metrics)

    cv_tr_mean = metrics[:,1].mean()
    cv_tr_std  = metrics[:,1].std()
    cv_va_mean = metrics[:,2].mean()
    cv_va_std  = metrics[:,2].std()

    oof_auc = roc_auc_score(input_y, oof)

    print("-"*20, f"result ({run_name})", "-"*20)
    print("[cv ] tr: {:.4f}+-{:.4f}, va: {:.4f}+-{:.4f}".format(
        cv_tr_mean, cv_tr_std,
        cv_va_mean, cv_va_std,
    ))
    print("[oof] {:.4f}".format(oof_auc))

    imp = imp.groupby("col")["imp"].agg(["mean", "std"]).reset_index()
    imp.columns = ["col", "imp", "imp_std"]

    log_text = (
        f"run: {run_name}\n"
        f"CV AUC_tr: {cv_tr_mean:.5f} ± {cv_tr_std:.5f}\n"
        f"CV AUC_va: {cv_va_mean:.5f} ± {cv_va_std:.5f}\n"
        f"OOF AUC: {oof_auc:.5f}"
    )

    # --- log_text も run ごとに保存 ---
    with open(f"{save_dir}/history.txt", "w") as f:
        f.write(log_text)

    return imp, metrics, oof, log_text

In [22]:
import os
import re

BASE_DIR = "/kaggle/working/runs"

def get_next_run_name(base_dir=BASE_DIR):
    os.makedirs(base_dir, exist_ok=True)

    runs = [d for d in os.listdir(base_dir) if re.match(r"run\d+", d)]

    if not runs:
        return "run0"

    nums = [int(re.search(r"run(\d+)", r).group(1)) for r in runs]
    next_id = max(nums) + 1

    return f"run{next_id}"


In [23]:
import re
import glob

def parse_log(text):
    tr = float(re.search(r"CV AUC_tr: ([0-9.]+)", text).group(1))
    va = float(re.search(r"CV AUC_va: ([0-9.]+)", text).group(1))
    oof = float(re.search(r"OOF AUC: ([0-9.]+)", text).group(1))
    return tr, va, oof

results = []

for path in sorted(glob.glob("/kaggle/working/runs/run*//history.txt")):
    text = open(path).read()
    tr, va, oof = parse_log(text)
    results.append([path, tr, va, oof])

pd.DataFrame(results, columns=["run", "AUC_tr", "AUC_va", "AUC_oof"])

,run,AUC_tr,AUC_va,AUC_oof


In [24]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001265 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 205
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000107 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 206
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 7
[LightGBM] [Info] [binary:BoostFro

In [25]:
#特徴量を追加
df_train["FamilySize"] = df_train["SibSp"] + df_train["Parch"] + 1
df_test["FamilySize"] = df_test["SibSp"] + df_test["Parch"] + 1

df_train["IsAlone"] = (df_train["FamilySize"] == 1).astype(int)
df_test["IsAlone"] = (df_test["FamilySize"] == 1).astype(int)

In [26]:
#データセットを作成
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

for col in ["Sex", "Embarked"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

print(x_train.shape, y_train.shape, id_train.shape)

(891, 9) (891,) (891,)


In [27]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000140 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 217
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000141 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 218
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 9
[LightGBM] [Info] [binary:BoostFro

In [28]:
imp.sort_values("imp", ascending=False, ignore_index=True)

,col,imp,imp_std
0,Sex,1019.906542,364.635872
1,Fare,390.838191,266.716636
2,Age,344.004708,220.361900
3,Pclass,338.535310,139.709421
4,FamilySize,86.829817,50.636016
5,Embarked,56.315918,60.013583
6,SibSp,16.181648,20.568703
7,Parch,10.564064,11.712698
8,IsAlone,0.947751,1.314360


In [29]:
df_train.drop(columns=["IsAlone"], inplace=True)
df_test.drop(columns=["IsAlone"], inplace=True)

In [30]:
#データセットを作成
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

for col in ["Sex", "Embarked"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

print(x_train.shape, y_train.shape, id_train.shape)

(891, 8) (891,) (891,)


In [31]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000139 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 215
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000142 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 216
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 8
[LightGBM] [Info] [binary:BoostFro

In [32]:
imp.sort_values("imp", ascending=False, ignore_index=True)

,col,imp,imp_std
0,Sex,1019.906542,364.635872
1,Fare,390.838191,266.716636
2,Age,344.004708,220.361900
3,Pclass,338.535310,139.709421
4,FamilySize,87.777567,51.792541
5,Embarked,56.315918,60.013583
6,SibSp,16.181648,20.568703
7,Parch,10.564064,11.712698


In [33]:
df_train["TicketGroup"] = df_train_raw.groupby("Ticket")["Ticket"].transform("count")
df_test["TicketGroup"] = df_test_raw.groupby("Ticket")["Ticket"].transform("count")

In [34]:
#データセットを作成
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

for col in ["Sex", "Embarked"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

print(x_train.shape, y_train.shape, id_train.shape)

(891, 9) (891,) (891,)


In [35]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000140 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 223
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000114 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 224
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 9
[LightGBM] [Info] [binary:BoostFro

In [36]:
imp.sort_values("imp", ascending=False, ignore_index=True)

,col,imp,imp_std
0,Sex,1177.362205,32.901809
1,Fare,423.514773,182.392654
2,Age,404.841803,119.548956
3,Pclass,380.015630,49.192995
4,FamilySize,98.194827,28.889870
5,Embarked,60.427539,42.196365
6,TicketGroup,37.108211,20.855351
7,Parch,16.813544,15.476993
8,SibSp,16.392512,11.832037


In [37]:
df_train["Deck"] = df_train_raw["Cabin"].str[0].fillna("U").astype("category")
df_test["Deck"] = df_test_raw["Cabin"].str[0].fillna("U").astype("category")

In [38]:
#データセットを作成
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

for col in ["Sex", "Embarked"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

print(x_train.shape, y_train.shape, id_train.shape)

(891, 10) (891,) (891,)


In [39]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000142 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 232
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000184 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 232
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 10
[LightGBM] [Info] [binary:BoostF

In [40]:
imp.sort_values("imp", ascending=False, ignore_index=True)

,col,imp,imp_std
0,Sex,1199.864367,22.162085
1,Age,526.776823,115.826739
2,Fare,508.815372,188.777136
3,Pclass,384.768633,54.953468
4,FamilySize,109.158333,17.908257
5,Embarked,70.592983,38.464921
6,TicketGroup,54.586995,11.491355
7,Deck,43.272864,13.344938
8,SibSp,22.693612,10.775309
9,Parch,20.446673,13.714346


In [41]:
def extract_title(df):
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
    
    # 主要4種類にまとめる（最小構造）
    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
    })

    df["Title"] = df["Title"].apply(lambda x: x if x in ["Mr", "Mrs", "Miss", "Master"] else "Other")
    return df

In [42]:
df_train["Title"] = extract_title(df_train_raw)["Title"]
df_test["Title"]  = extract_title(df_test_raw)["Title"]

In [43]:
#データセットを作成
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

for col in ["Sex", "Embarked", "Title"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

print(x_train.shape, y_train.shape, id_train.shape)

(891, 11) (891,) (891,)


In [44]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000180 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 238
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000136 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 238
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 11
[LightGBM] [Info] [binary:BoostF

In [45]:
imp.sort_values("imp", ascending=False, ignore_index=True)

,col,imp,imp_std
0,Sex,997.435995,144.510492
1,Fare,535.182841,253.641641
2,Age,474.628016,186.090970
3,Pclass,357.626681,77.472756
4,Title,180.705663,135.088205
5,FamilySize,109.855053,61.632998
6,TicketGroup,79.395364,27.021689
7,Embarked,68.278361,58.097589
8,Deck,30.887270,22.075997
9,SibSp,28.835055,29.026733


In [46]:
def extract_title(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")

    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
    })

    officer = ["Col", "Major", "Capt"]
    royalty = ["Lady", "Sir", "Countess"]

    df["Title"] = df["Title"].apply(
        lambda x:
            "Officer" if x in officer else
            "Royalty" if x in royalty else
            x if x in ["Mr", "Mrs", "Miss", "Master"] else
            "Other"
    )
    return df

In [47]:
df_train["Title"] = extract_title(df_train_raw)["Title"]
df_test["Title"]  = extract_title(df_test_raw)["Title"]

In [48]:
#データセットを作成
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

for col in ["Sex", "Embarked", "Title"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

print(x_train.shape, y_train.shape, id_train.shape)

(891, 11) (891,) (891,)


In [49]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000150 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 238
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000130 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 238
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 11
[LightGBM] [Info] [binary:BoostF

In [50]:
imp.sort_values("imp", ascending=False, ignore_index=True)

,col,imp,imp_std
0,Sex,996.312214,145.891513
1,Fare,533.443326,223.056595
2,Age,471.168701,165.220375
3,Pclass,356.637753,76.621147
4,Title,184.174371,131.927616
5,FamilySize,112.056080,63.974943
6,TicketGroup,81.749704,29.676916
7,Embarked,68.337756,55.466904
8,Deck,32.078129,21.288336
9,SibSp,25.286770,21.736265


In [51]:
def extract_title(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")

    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
    })

    officer = ["Col", "Major", "Capt"]
    royalty = ["Lady", "Sir", "Countess"]

    df["Title"] = df["Title"].apply(
        lambda x:
            "Officer" if x in officer else
            "Royalty" if x in royalty else
            "Dr" if x == "Dr" else
            "Rev" if x == "Rev" else
            x if x in ["Mr", "Mrs", "Miss", "Master"] else
            "Common"
    )
    return df

In [52]:
df_train["Title"] = extract_title(df_train_raw)["Title"]
df_test["Title"]  = extract_title(df_test_raw)["Title"]

In [53]:
#データセットを作成
x_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]
id_train = df_train["PassengerId"]

x_test = df_test.drop(columns=["PassengerId"])
id_test = df_test["PassengerId"]

for col in ["Sex", "Embarked", "Title"]:
    x_train[col] = x_train[col].astype("category")
    x_test[col]  = x_test[col].astype("category")

print(x_train.shape, y_train.shape, id_train.shape)

(891, 11) (891,) (891,)


In [54]:
run_name = get_next_run_name()
imp, metrics, oof, log_text = train_cv(x_train, y_train, params, run_name=run_name)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000156 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 240
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000200 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 240
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.384292 -> initscore=-0.471371
[LightGBM] [Info

In [55]:
imp.sort_values("imp", ascending=False, ignore_index=True)

,col,imp,imp_std
0,Sex,1022.368923,137.040788
1,Fare,523.288999,165.307748
2,Age,486.911871,72.355577
3,Pclass,358.602567,65.509961
4,Title,196.788323,122.387829
5,FamilySize,113.224862,48.484017
6,TicketGroup,88.697703,28.955993
7,Embarked,70.585681,48.059337
8,Deck,39.659648,15.304449
9,Parch,26.096683,9.747156


In [56]:
pd.DataFrame(results, columns=["run", "AUC_tr", "AUC_va", "AUC_oof"])

,run,AUC_tr,AUC_va,AUC_oof


In [57]:
import lightgbm as lgb
import numpy as np
import pandas as pd

def predict_lgb(input_x,
                input_id,
                run_name,
                list_nfold=[0,1,2,3,4]):
    
    model_dir = f"/kaggle/working/runs/{run_name}"
    
    pred = np.zeros((len(input_x), len(list_nfold)))
    
    for nfold in list_nfold:
        print("-"*20, f"fold {nfold}", "-"*20)
        
        # ★ LightGBM のテキストモデルを読み込む
        fname_lgb = f"{model_dir}/model_fold{nfold}.txt"
        model = lgb.Booster(model_file=fname_lgb)
        
        # LightGBM Booster の predict
        pred[:, nfold] = model.predict(input_x)
    
    pred_mean = pred.mean(axis=1)
    
    pred_df = pd.concat([
        input_id.reset_index(drop=True),
        pd.DataFrame({"pred": pred_mean})
    ], axis=1)
    
    print("Done.")
    return pred_df

In [58]:
pred_df = predict_lgb(
    input_x=x_test,
    input_id=id_test,
    run_name="run7"
)

-------------------- fold 0 --------------------
-------------------- fold 1 --------------------
-------------------- fold 2 --------------------
-------------------- fold 3 --------------------
-------------------- fold 4 --------------------
Done.


In [59]:
submission = pred_df.copy()
submission["Survived"] = (submission["pred"] > 0.5).astype(int)
submission = submission[["PassengerId", "Survived"]]
submission.to_csv("submission.csv", index=False)

In [60]:
print(submission.shape)
display(submission.head())

(418, 2)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
